<a href="https://colab.research.google.com/github/ymrow/ChallengeAluraAgente/blob/main/Agente_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-google-genai
!pip install -q langchain-text-splitters
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 7.1 MB/s eta 0:00:00


In [3]:
import os

from google.colab import userdata

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS

from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_google_genai import ChatGoogleGenerativeAI

import gradio as gr

print("Librerías cargadas")

/tmp/ipykernel_1292/2509057048.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Librerías cargadas


In [8]:
RUTA_DOCUMENTOS = "/content/documentos"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

MODELO_EMBEDDINGS = "sentence-transformers/all-MiniLM-L6-v2"

MODELO_GEMINI = "gemini-3.1-flash-lite"

TOP_K = 5

print("Configuración lista")

Configuración lista


In [9]:
documentos = []

if not os.path.exists(RUTA_DOCUMENTOS):
    raise Exception(f"No existe la carpeta {RUTA_DOCUMENTOS}")

print("Archivos encontrados:\n")

for archivo in os.listdir(RUTA_DOCUMENTOS):

    if archivo.lower().endswith(".pdf"):

        print("•", archivo)

        loader = PyPDFLoader(
            os.path.join(RUTA_DOCUMENTOS, archivo)
        )

        documentos.extend(loader.load())

print()

print(" Total de páginas:", len(documentos))

Archivos encontrados:

• Politica de reembolsos y devoluciones.pdf
• Preguntas Frecuentes sobre Métodos de Pago.pdf
• Programa de afiliados.pdf
• Manual de Garantía de Productos.pdf
• Guía de Tiempos y Costos de Envío.pdf

 Total de páginas: 57


In [10]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

chunks = splitter.split_documents(documentos)

print("Fragmentos creados:", len(chunks))

Fragmentos creados: 108


In [11]:
embeddings = HuggingFaceEmbeddings(
    model_name=MODELO_EMBEDDINGS
)

print("Embeddings cargados")

/tmp/ipykernel_1292/4068285478.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings cargados


In [12]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Base vectorial creada")

Base vectorial creada


In [13]:
api_key = userdata.get("GEMINI_API_KEY")

os.environ["GOOGLE_API_KEY"] = api_key

llm = ChatGoogleGenerativeAI(
    model=MODELO_GEMINI,
    temperature=0
)

print(" Gemini configurado")

 Gemini configurado


In [14]:
def limpiar_respuesta(respuesta):

    contenido = respuesta.content

    if isinstance(contenido, list):

        texto = ""

        for bloque in contenido:

            if isinstance(bloque, dict):

                if bloque.get("type") == "text":

                    texto += bloque.get("text", "")

        return texto.strip()

    if isinstance(contenido, str):

        return contenido.strip()

    return str(contenido)

In [16]:
def responder_usuario(pregunta):

    documentos = vectorstore.similarity_search(
        pregunta,
        k=TOP_K
    )

    contexto = "\n\n".join(
        doc.page_content
        for doc in documentos
    )

    prompt = f"""
Eres el asistente virtual de atención al cliente de BimBam Buy.

Tu objetivo es ayudar al usuario respondiendo preguntas
sobre la información disponible en los documentos.

Debes responder únicamente utilizando la información del contexto.

Reglas:

- Responde de forma clara y cordial.
- Si la información es parcial, explícalo.
- No inventes datos.
- Si existe información relacionada, úsala.
- Si realmente no existe información suficiente, indícalo.

CONTEXTO

{contexto}

PREGUNTA

{pregunta}

RESPUESTA
"""

    respuesta = llm.invoke(prompt)

    return limpiar_respuesta(respuesta)

In [17]:
respuesta = responder_usuario(
    "¿Cómo puedo acceder al envío gratis?"
)

print(respuesta)

¡Hola! Es un gusto saludarte. Como asistente virtual de BimBam Buy, con gusto te informo sobre cómo funciona el envío gratis en nuestra plataforma:

El envío gratis puede estar disponible bajo condiciones promocionales o al alcanzar montos mínimos de compra, los cuales son definidos según el país y la campaña vigente.

Para acceder a este beneficio, ten en cuenta lo siguiente:
* **Visualización:** El envío gratis se mostrará claramente antes de realizar el pago.
* **Condiciones:** Puede aplicar únicamente a categorías o regiones determinadas.
* **Vigencia:** Este beneficio puede tener límites de tiempo o de stock.
* **Consideración importante:** El envío gratis no cubre necesariamente los costos de reenvío si estos son atribuibles al cliente.

Te recomiendo revisar las promociones vigentes antes de finalizar tu compra para verificar si tu pedido califica para este beneficio. ¡Quedo a tu disposición si tienes alguna otra duda!


In [18]:
interfaz = gr.Interface(

    fn=responder_usuario,

    inputs=gr.Textbox(
        lines=2,
        placeholder="Escribe tu consulta..."
    ),

    outputs=gr.Textbox(
        label="Respuesta"
    ),

    title="🛒 Asistente Virtual BimBam Buy",

    description="""
Consulta información sobre:

• Pagos
• Envíos
• Reembolsos
• Devoluciones
• Programa de afiliados
• Promociones
"""
)

print("✅ Interfaz creada")

✅ Interfaz creada


In [19]:
interfaz.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e8ee47e345be2cd673.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
